# Efficiency plotting 

Copied most functionality from Moon's https://github.com/wjdanswjddl/cafpyana/blob/release/numucc_1p0pi/analysis_village/numucc_1p0pi/notebooks/event_selection.ipynb

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# print avaialbe memory
import psutil
print(psutil.virtual_memory())

svmem(total=1035267956736, available=980369760256, percent=5.3, used=48283713536, free=666073169920, active=66384392192, inactive=191574220800, buffers=7766016, cached=320903307264, shared=105230336, slab=104129732608)


In [3]:
import pandas as pd
import numpy as np
import sys
from os import path, makedirs
from datetime import datetime
import pickle

# local imports
# sys.path.append('../../../')
sys.path.append('/nashome/m/micarrig/sbnd/nueCCNp/cafpyana/') # absolute path for running on EAF
from analysis_village.nueNp0Pi.variable_configs import VariableConfig
from analysis_village.nueNp0Pi.categories import *
from analysis_village.nueNp0Pi.utils import *
from analysis_village.nueNp0Pi.files_config import *
from analysis_village.nueNp0Pi.makedf.selections import *
from pyanalib.split_df_helpers import *
from pyanalib.pandas_helpers import *
from pyanalib.covariance import *

import matplotlib.pyplot as plt 
from matplotlib.patches import Patch

plt.style.use("/nashome/m/micarrig/sbnd/nueCCNp/cafpyana/analysis_village/nueNp0Pi/notebooks/presentation.mplstyle")

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
# turn off RuntimeWarning
warnings.filterwarnings("ignore", category=RuntimeWarning)

os.environ["CAFPYANA_LOG_LEVEL"] = "DEBUG"

In [4]:
USE_BATCHED_WORKFLOW = True  # False → legacy in-memory cells below (small tests only)

from analysis_village.nueNp0Pi.event_selection_batched import (
    EventSelectionBatchedConfig,
    discover_jobs,
    run_full,
    show_saved_plots,
    default_event_selection_batched_work_root,
)
from analysis_village.nueNp0Pi.dataset_locations import PLOTS_BASE

today_str = datetime.now().strftime("%Y%m%d")
today_str = "thesis"
syst_tag = ""

# Optional overrides (None → dated work dir under /exp/sbnd/data/users/$USER/...)
batch_work_base = f"/exp/sbnd/data/users/micarrig/xsec/nueNp0Pi/event_selection-batched-{today_str}"
batch_plots_dir = path.join(PLOTS_BASE, f"event_selection-{syst_tag}-{today_str}")

batch_cfg = EventSelectionBatchedConfig(
    work_base=batch_work_base or default_event_selection_batched_work_root(today_str),
    plots_dir=batch_plots_dir,
    max_job_bytes=int(1.0 * 1024**3),  # 1 GiB per job
    mc_univ_syst=(), #("Flux", "G4", "GENIE"),
    skip_existing_batches=False,
    aggregate_only=False,
    skip_aggregate=False,
    save_fig=True,
    show_fig=False,
    # max_files_per_sample=1,  # smoke test: one file per sample
)

if USE_BATCHED_WORKFLOW:
    records, jobs, manifest_path = discover_jobs(batch_cfg)
    print(f"Batched workflow: {len(jobs)} job(s)  manifest={manifest_path}")
    for j in jobs[:8]:
        print(f"  {j.sample} {j.tag}: {len(j.files)} file(s), {j.total_bytes / (1024**3):.3f} GiB")
    if len(jobs) > 8:
        print(f"  ... and {len(jobs) - 8} more")

[survey_files] mc: looking in /exp/sbnd/data/users/micarrig/nueNp0Pi/selection_test/*.df
[survey_files] mc: found 1 file(s)
[batched] sample=mc  files=1  total=6.81 GiB  glob=/exp/sbnd/data/users/micarrig/nueNp0Pi/selection_test/*.df
[batched] 1 job(s) under size budget
  mc batch_0000: 1 file(s), 6.813 GiB
Batched workflow: 1 job(s)  manifest=/exp/sbnd/data/users/micarrig/xsec/nueNp0Pi/event_selection-batched-thesis/manifest.json
  mc batch_0000: 1 file(s), 6.813 GiB


In [5]:
batch_result = None

if USE_BATCHED_WORKFLOW:
    batch_result = run_full(batch_cfg)

    save_fig_dir = str(batch_result.plots_dir)
    save_fig = batch_cfg.save_fig
    show_plot = batch_cfg.show_fig
    pot_str = batch_result.pot_str
    data_tot_pot = batch_result.data_pot
    merged_payload = batch_result.merged_payload

    print("Batched workflow complete.")
    print("  batches:", batch_result.batches_dir)
    print("  plots  :", batch_result.plots_dir)
    print("  manifest:", batch_result.manifest_path)
    print("  POT    :", pot_str)

    # Uncomment to preview PNGs inline (can be slow for many plots):
    # show_saved_plots(batch_result.plots_dir, max_images=20)
else:
    print("USE_BATCHED_WORKFLOW=False — run legacy data-loading cells below.")

[survey_files] mc: looking in /exp/sbnd/data/users/micarrig/nueNp0Pi/selection_test/*.df
[survey_files] mc: found 1 file(s)
[batched] sample=mc  files=1  total=6.81 GiB  glob=/exp/sbnd/data/users/micarrig/nueNp0Pi/selection_test/*.df
[batched] 1 job(s) under size budget
  mc batch_0000: 1 file(s), 6.813 GiB
[batched] WORK_BASE=/exp/sbnd/data/users/micarrig/xsec/nueNp0Pi/event_selection-batched-thesis
[batched] BATCHES_DIR=/exp/sbnd/data/users/micarrig/xsec/nueNp0Pi/event_selection-batched-thesis/batches
[batched] sample=mc batch_0000  files=1  size=6.813 GiB
[batch_map] sample=mc job=batch_0000 files=1 → /exp/sbnd/data/users/micarrig/xsec/nueNp0Pi/event_selection-batched-thesis/batches/mc__batch_0000.pkl


[DEBUG] cafpyana.analysis_village.nueNp0Pi.event_selection_pipeline_def: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.event_selection_pipeline_def:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.event_selection_pipeline_def:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.event_selection_pipeline_def:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.event_selection_pipeline_def:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.event_selection_pipeline_def:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.event_selection_pipeline_def:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,bre

[batch_map] done  n_evt=56335  pot=1.425e+19  per_file=[1.424814787237249e+19]
[batched] sample=mc wrote 1 batch pickle(s)
[aggregate] sample=mc -> 1 chunks
[aggregate] sample=data -> 0 chunks
[aggregate] sample=intime -> 0 chunks
[aggregate] sample=offbeam -> 0 chunks
[aggregate] sample=dirt -> 0 chunks
[batched] aggregating 1 batches for sample=mc


[INFO] cafpyana.pyanalib.chunked_selection: aggregating 1 chunk file(s)
[INFO] cafpyana.pyanalib.chunked_selection: merging samples: ['mc']
[INFO] cafpyana.pyanalib.chunked_selection: applied global exposure scales: {'scale_mc': 1.0, 'scale_dirt': 1.0, 'scale_intime': 0.0, 'scale_offbeam': 0.0}


[aggregate] exposure totals: data_pot=0.000e+00 bnb_gates=0.000e+00 mc_pot=1.425e+19 dirt_pot=0.000e+00 intime_gates=0.000e+00 offbeam_gates=0.000e+00
[batched] applied global scales: {'scale_mc': 1.0, 'scale_dirt': 1.0, 'scale_intime': 0.0, 'scale_offbeam': 0.0}
[batched] mc_pot=1.425e+19 -> POT label=1.42$\times 10^{19}$
[batched] systematics disk root: None


[DEBUG] cafpyana.analysis_village.nueNp0Pi.event_selection_pipeline_def: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.event_selection_pipeline_def:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.event_selection_pipeline_def:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.event_selection_pipeline_def:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.event_selection_pipeline_def:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.event_selection_pipeline_def:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.event_selection_pipeline_def:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,bre

[aggregate] final selection purity: 87.64% (weighted)  87.64% (raw counts, notebook-style)
[aggregate] wrote /exp/sbnd/data/users/micarrig/plots/numucc1p0pi/event_selection--thesis/eff_dict.pkl
[batched] wrote /exp/sbnd/data/users/micarrig/plots/numucc1p0pi/event_selection--thesis/merged_histdata.pkl
[batched] DONE plots -> /exp/sbnd/data/users/micarrig/plots/numucc1p0pi/event_selection--thesis
Batched workflow complete.
  batches: /exp/sbnd/data/users/micarrig/xsec/nueNp0Pi/event_selection-batched-thesis/batches
  plots  : /exp/sbnd/data/users/micarrig/plots/numucc1p0pi/event_selection--thesis
  manifest: /exp/sbnd/data/users/micarrig/xsec/nueNp0Pi/event_selection-batched-thesis/manifest.json
  POT    : 1.42$\times 10^{19}$
